In [ ]:
import optuna
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.datasets import cifar100
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import pandas as pd
import datetime
import io
import contextlib


In [167]:
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))


Num GPUs Available: 1


In [ ]:
NUM_CLASSES = 100
INPUT_SHAPE = (32, 32, 3)
MAX_EPOCHS = 50
VALIDATION_SPLIT = 0.2

In [ ]:
(x_train, y_train), (x_test, y_test) = cifar100.load_data(label_mode='fine')

# Normalize pixel values
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_test_cat = to_categorical(y_test, NUM_CLASSES)

In [ ]:
# Global list to store results
trial_log = []


def objective(trial):
    K.clear_session()

    # 🎯 Hyperparameter suggestions
    lr = trial.suggest_loguniform("lr", 1e-5, 1e-2)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop"])
    dropout_rate = trial.suggest_float("dropout", 0.2, 0.5)

    # Filters
    filters_1 = trial.suggest_categorical("filters_1", [64, 128, 256, 512])  # always included
    filters_2 = trial.suggest_categorical("filters_2", [0, 64, 128, 256, 512])
    filters_3 = trial.suggest_categorical("filters_3", [0, 64, 128, 256, 512])

    dense_units = trial.suggest_categorical("dense_units", [128, 256, 512])
    batch_size = trial.suggest_categorical("batch_size", [64, 128])

    optimizer = Adam(learning_rate=lr) if optimizer_name == "Adam" else RMSprop(learning_rate=lr)

    # 🧱 Build model
    model = Sequential()

    # Conv Block 1 (always included)
    model.add(Conv2D(filters_1, (3, 3), padding="same", activation='relu', input_shape=INPUT_SHAPE))
    model.add(BatchNormalization())
    model.add(Conv2D(filters_1, (3, 3), padding="same", activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(dropout_rate))

    # Conv Block 2 (optional)
    if filters_2 > 0:
        model.add(Conv2D(filters_2, (3, 3), padding="same", activation='relu'))
        model.add(BatchNormalization())
        model.add(Conv2D(filters_2, (3, 3), padding="same", activation='relu'))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        model.add(Dropout(dropout_rate))

    # Conv Block 3 (optional)
    if filters_3 > 0:
        model.add(Conv2D(filters_3, (3, 3), padding="same", activation='relu'))
        model.add(BatchNormalization())
        model.add(Conv2D(filters_3, (3, 3), padding="same", activation='relu'))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        model.add(Dropout(dropout_rate))

    # Dense block
    model.add(Flatten())
    model.add(Dense(dense_units, activation='relu'))
    model.add(Dropout(dropout_rate))
    model.add(Dense(NUM_CLASSES, activation='softmax'))

    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    # Callbacks
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=0)

    # Train the model
    history = model.fit(
        x_train, y_train_cat,
        validation_split=VALIDATION_SPLIT,
        epochs=MAX_EPOCHS,
        batch_size=batch_size,
        verbose=0,
        callbacks=[early_stopping, reduce_lr]
    )

    val_acc = history.history['val_accuracy'][-1]
    test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
    
    with io.StringIO() as buf, contextlib.redirect_stdout(buf):
        model.summary()
        model_summary_str = buf.getvalue()


    trial_log.append({
        'trial': trial.number,
        'lr': lr,
        'optimizer': optimizer_name,
        'dropout': dropout_rate,
        'filters_1': filters_1,
        'filters_2': filters_2,
        'filters_3': filters_3,
        'dense_units': dense_units,
        'batch_size': batch_size,
        'val_accuracy': val_acc,
        'test_accuracy': test_acc,
        'epochs_ran': len(history.history['accuracy']),
        'timestamp': datetime.datetime.now().isoformat(),
        'model_summary': model_summary_str 
    })

    return val_acc


In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

# Save logs
df = pd.DataFrame(trial_log)
df.to_csv("optuna_trial_results.csv", index=False)

# Print the best
print("Best hyperparameters:")
print(study.best_params)
print(f"Best validation accuracy: {study.best_value:.4f}")
